In [38]:
import io
import pandas as pd
import numpy as np
from zipfile import ZipFile

#from google.colab import files

import ipywidgets as widgets
from IPython.display import display, clear_output

from src.algorithm import run_assignments, ModuleAssigner

from src.data_loading import (
    get_formatted_module_data,
    load_students,
    validate_module_data,
    validate_module_rankings_data,
    validate_module_group_preferences_data,
    validate_module_assignments_data,
    check_ranking_and_group_ids_match
)

In [39]:
modules = None
students = None

module_df = None
rankings_df = None
group_preferences_df = None
prior_allocations_df = None

module_groups = None
semesters = None

required_credits_widget = None

group_min_widgets = {}
group_max_widgets = {}

semester_min_widgets = {}
semester_max_widgets = {}

best_assignment = None

## Step 1 - Data Loading

In [ ]:
#
# DATAFILE SELECTION
#

module_upload = widgets.FileUpload(
    accept=".csv",
    multiple=False,
    description="Modules"
)

rankings_upload = widgets.FileUpload(
    accept=".csv",
    multiple=False,
    description="Rankings"
)

group_upload = widgets.FileUpload(
    accept=".csv",
    multiple=False,
    description="Group Prefs"
)

programme_modules_upload = widgets.FileUpload(
    accept=".csv",
    multiple=False,
    description="Programme Modules"
)

prior_upload = widgets.FileUpload(
    accept=".csv",
    multiple=False,
    description="Prior Allocs"
)

display(
    widgets.VBox([
        widgets.HTML("<h3>Upload Input Files</h3>"),
        module_upload,
        rankings_upload,
        group_upload,
        prior_upload
    ])
)


#
# DATA LOADING / PREPROCESSING
#

def load_uploaded_csv(upload_widget):
    ## Read the data from the given FileUpload widget into a Pandas dataframe
    if len(upload_widget.value) == 0:
        return None
    uploaded_file = upload_widget.value[0]
    return pd.read_csv(io.BytesIO(uploaded_file["content"]))

def on_load_clicked(button):

    global modules
    global students
    global programmes

    global module_df
    global rankings_df
    global group_preferences_df
    global prior_allocations_df

    global module_groups
    global semesters

    with load_output:

        clear_output()

        print("Loading data...")

        try:
            # Load the selected data files into Pandas dataframes
            module_df = load_uploaded_csv(module_upload)
            rankings_df = load_uploaded_csv(rankings_upload)
            group_preferences_df = load_uploaded_csv(group_upload)
            prior_allocations_df = load_uploaded_csv(prior_upload)

            if module_df is None:
                raise Exception("Module file not uploaded")

            if rankings_df is None:
                raise Exception("Rankings file not uploaded")

            if group_preferences_df is None:
                raise Exception("Group preferences file not uploaded")
            
            ranking_errors = validate_module_rankings_data(rankings_df)
            if len(ranking_errors) > 0:
                print("RANKING FILE ERRORS")
                for e in ranking_errors:
                    print(f"\t{e}")
                return            

            # Check for invalid content in the loaded data
            programmes = list(rankings_df["programme"].unique())
            module_errors = validate_module_data(module_df, programmes)
            if len(module_errors) > 0:
                print("MODULE FILE CONTAINS ERRORS")
                for e in module_errors:
                    print(f"\t{e}")
                return

            group_errors = validate_module_group_preferences_data(group_preferences_df, list(module_df["module_group"].unique()))
            if len(group_errors) > 0:
                print("GROUP PREFERENCE FILE ERRORS")
                for e in group_errors:
                    print(f"\t{e}")
                return

            # Check for invalid content in the previous module allocations (if loaded)
            if not prior_allocations_df is None:
                prior_allocation_errors = validate_module_assignments_data(prior_allocations_df)
                if len(prior_allocation_errors) > 0:
                    print("PRIOR ALLOCATIONS FILE CONTAINS ERRORS")
                    for e in module_errors:
                        print(f"\t{e}")
                    return

            # Check for missing students in the preferences files
            missing_rankings, missing_group_prefs = check_ranking_and_group_ids_match(rankings_df, group_preferences_df)
            if len(missing_rankings) > 0:
                print("Students missing from rankings:")
                print(f"\t{missing_rankings}")
                return

            if len(missing_group_prefs) > 0:
                print("Students missing from group preferences:")
                print(f"\t{missing_group_prefs}")
                return
            
            # Format the loaded data for use in the allocation algorithm
            modules, module_groups, semesters, required_modules_not_found, mutually_excluded_modules_not_found = get_formatted_module_data(module_df, list(rankings_df["programme"].unique()))
            students, students_missing_ranks, students_missing_ids, students_missing_programmes, missing_modules = load_students(rankings_df, group_preferences_df, modules)
            


            print("Data loaded successfully")
            print(f"\tStudents: {len(students)}")
            print(f"\tModules: {len(modules)}")
            print(f"\tModule Groups: {len(module_groups)} - {module_groups}")
            print(f"\tSemesters: {len(semesters)} - {[str(s) for s in semesters]}")

            if len(students_missing_ids) > 0:
                print("Warning: Some students had IDs missing in the Module Rankings file")
                print(f"Students: {students_missing_ids}")

            if len(students_missing_programmes) > 0:
                print("Warning: Some students had no listed programme of study")
                print(f"Students: {students_missing_programmes}")

            if len(students_missing_ranks) > 0:
                print("Warning: Some students had missing preference ratings for some modules in the Module Rankings file.\nUnrated modules will be given the lowest possible preference for these students (if those modules are allowed, depending on the student's programme).")
                print(f"Students: {students_missing_ranks}")

            if len(missing_modules) > 0:
                print("Warning: Students did not provide preferences for some modules:")
                print(f"Modules: {missing_modules}")

            
        except Exception as e:
            print("UNEXPECTED ERROR")
            print(e)


load_button = widgets.Button(
    description="Load Data",
    button_style="info"
)
load_output = widgets.Output()
display(load_button)
display(load_output)

load_button.on_click(on_load_clicked)


Button(button_style='info', description='Load Data', style=ButtonStyle())

Output()

## Step 2 - Constraints

In [44]:
#
# Credit Constraints UI
#
INT_TEXT_STYLE = {"description_width":'225px'}

group_min_widgets = {}
group_max_widgets = {}

semester_min_widgets = {}
semester_max_widgets = {}

group_widgets = []
semester_widgets = []

programme_to_assign_widget = None

halt_after_widget = None
repetitions_widget = None
seed_widget = None
check_constraints_widget = None

step_check_output = widgets.Output()
display(step_check_output)

if module_groups == None or semesters == None:
    with step_check_output:
        print("Module group data not found. Did you finish Step 1 before running Step 2?")
        
else:
    
    required_credits_widget = widgets.IntText(
        value=6,
        description="Required Credits Per Student",
        style=INT_TEXT_STYLE
    )

    for group in module_groups:
        min_widget = widgets.IntText(value=0, style=INT_TEXT_STYLE, description=f"{group} Min")
        max_widget = widgets.IntText(value=3, style=INT_TEXT_STYLE, description=f"{group} Max")

        group_min_widgets[group] = min_widget
        group_max_widgets[group] = max_widget

        group_widgets.append(min_widget)
        group_widgets.append(max_widget)

    for semester in semesters:
        min_widget = widgets.IntText(value=0, style=INT_TEXT_STYLE, description=f"Semester {semester} (Min)")
        max_widget = widgets.IntText(value=3, style=INT_TEXT_STYLE, description=f"Semester {semester} (Max)")

        semester_min_widgets[semester] = min_widget
        semester_max_widgets[semester] = max_widget

        semester_widgets.append(min_widget)
        semester_widgets.append(max_widget)

    max_module_rating_to_allow_widget = widgets.IntText(value=7, style=INT_TEXT_STYLE, description=f"Worst Module Preference to Allow")
    programme_to_assign_widget = widgets.Dropdown(options=programmes, value=programmes[0],disabled=False)

    display(
        widgets.VBox([
            widgets.HTML("<h3>Allocation Constraints</h3>"),
            required_credits_widget,
            widgets.HTML("<b>Module Group Credit Requirements</b>"),
            *group_widgets,
            widgets.HTML("<b>Semester Credit Requirements</b>"),
            *semester_widgets,
            widgets.HTML("<b>Worst (Maximum) Module Preference to Allow</b>"),
            max_module_rating_to_allow_widget,
            widgets.HTML("<b>Programme to Assign</b>"),
            programme_to_assign_widget
        ])
    )

    #
    # Allocation Settings UI
    #

    halt_after_widget = widgets.IntText(
        value=3,
        description="Stop After (N Modules Per Student)",
        style=INT_TEXT_STYLE
    )

    repetitions_widget = widgets.IntText(
        value=10,
        description="Optimisation Iterations",
        style=INT_TEXT_STYLE
    )

    seed_widget = widgets.IntText(
        value=8194761,
        description="Random Seed",
        style=INT_TEXT_STYLE
    )

    check_constraints_widget = widgets.Checkbox(
        value=True,
        description="Check Constraints"
    )

    display(
        widgets.VBox([
            widgets.HTML("<h3>Allocation Settings</h3>"),
            halt_after_widget,
            repetitions_widget,
            seed_widget,
            check_constraints_widget
        ])
    )



Output()

## Step 3 - Allocation

In [ ]:
from copy import deepcopy

step_3_pre_check_output = widgets.Output()
display(step_3_pre_check_output)


if (group_min_widgets == {} or
    group_max_widgets == {} or
    semester_min_widgets == {} or
    semester_max_widgets == {} or
    group_widgets == [] or
    semester_widgets == [] or
    programme_to_assign_widget == None or
    halt_after_widget == None or
    repetitions_widget == None or
    seed_widget == None or
    check_constraints_widget == None):
    
    with step_3_pre_check_output:
        print("Constraint data not found. Did you finish Step 2 before running Step 3?")
        
else:
    
    best_assignments_per_programme:dict[str, ModuleAssigner | None] = {p:None for p in programmes}
    global previous_modules_lists 
    previous_modules_lists = [deepcopy(modules)]
    ordered_best_assignments = []

    run_button = widgets.Button(
        description="Run Allocation",
        button_style="success",
        icon="play"
    )

    progress_bar = widgets.IntProgress(
        value=0,
        min=0,
        max=100,
        description="Progress:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="600px")
    )

    run_output = widgets.Output()

    display(run_button)
    display(progress_bar)
    display(run_output)


    def on_run_clicked(button):

        
        global best_assignment

        with run_output:

            clear_output()

            print("Preparing allocation...")

            if students is None:
                print("No student data loaded")
                return

            if modules is None:
                print("No module data loaded")
                return

            required_credits = required_credits_widget.value

            min_credits_per_group = {
                group: group_min_widgets[group].value
                for group in module_groups
            }

            max_credits_per_group = {
                group: group_max_widgets[group].value
                for group in module_groups
            }

            min_credits_per_semester = {
                semester: semester_min_widgets[semester].value
                for semester in semesters
            }

            max_credits_per_semester = {
                semester: semester_max_widgets[semester].value
                for semester in semesters
            }

            max_module_rating_to_allow = max_module_rating_to_allow_widget.value
            
            best_assignment = None
            new_last_modified_modules = None
            last_modified_modules = previous_modules_lists[-1]

            progress_bar.max = repetitions_widget.value
            progress_bar.value = 0

            for repetition in range(repetitions_widget.value):

                best_assignment, new_last_modified_modules = run_assignments(
                    programme_to_assign=programme_to_assign_widget.value,
                    repetition = repetition,
                    previous_assignment = best_assignment,
                    halt_after_n_assignments = halt_after_widget.value,
                    check_constraints = check_constraints_widget.value,
                    loaded_module_assignments = prior_allocations_df,
                    students = students,
                    modules = last_modified_modules,
                    required_credits_per_student = required_credits,
                    max_credits_per_group = max_credits_per_group,
                    max_credits_per_semester = max_credits_per_semester,
                    min_credits_per_group = min_credits_per_group,
                    min_credits_per_semester = min_credits_per_semester,
                    max_module_rating_to_allow = max_module_rating_to_allow,
                    random_seed = seed_widget.value + repetition
                )

                progress_bar.value = repetition + 1

            print("Allocation complete")

            if best_assignment is None:
                print("No valid assignment found")
                return
            else:
                previous_modules_lists.append(new_last_modified_modules)
            

            best_assignments_per_programme[programme_to_assign_widget.value] = best_assignment
            ordered_best_assignments.append(best_assignment)


            print(f"Best assignment for programme {programme_to_assign_widget.value} generated successfully")
    
    run_button.on_click(on_run_clicked)


    results_button = widgets.Button(
    description="Show Results",
    button_style="info"
    )

    results_output = widgets.Output()

    display(results_button)
    display(results_output)

    def on_results_clicked(button):

        with results_output:

            clear_output()

            if best_assignment is None:

                print(
                    "Run allocation first"
                )

                return

            scores = (
                best_assignment
                .get_assignment_satisfaction_scores()
            )

            print(
                f"Mean satisfaction: "
                f"{np.nanmean(scores):.3f}"
            )

            print(
                f"Median satisfaction: "
                f"{np.nanmedian(scores):.3f}"
            )

            print()

            print(
                "Assigned credits summary"
            )

            credits = (
                best_assignment
                .get_assigned_credits_totals()
            )

            print(
                f"Min credits: {credits.min()}"
            )

            print(
                f"Max credits: {credits.max()}"
            )

            print(
                f"Mean credits: {credits.mean():.1f}"
            )

            print()
            print(
                "Student assignments"
            )

            for programme in programmes:
                if best_assignments_per_programme[programme] != None:
                    display(
                        best_assignments_per_programme[programme]
                        .get_all_assigned_modules(module_df)
                    )

    results_button.on_click(
        on_results_clicked
    )

Output()

## Step 4 - Results

In [43]:
download_button = widgets.Button(
    description="Download Results",
    button_style="warning",
    icon="download"
)

download_output = widgets.Output()

display(download_button)
display(download_output)

def on_download_clicked(button):

    programmes_missing = []
    for programme in programmes:
        if best_assignments_per_programme[programme] is None:
            programmes_missing.append(programme)


    with download_output:

        clear_output()

        if len(programmes_missing) > 0:

            print("Some programmes were not assigned modules:")
            print(programmes_missing)
            print("Run assignment for all programmes, or remove unneeded programmes from the data files")
            return

        print(
            "Creating results zip..."
        )

        zip_filename = (
            "module_allocation_results.zip"
        )

        module_ids:list[str] = []
        assigned_students_data:list[pd.DataFrame] = []
        assignment_summary_df: pd.DataFrame = pd.DataFrame()
        excess_requests_df: pd.DataFrame = pd.DataFrame()

        for best_assignment in best_assignments_per_programme.values():

            b_module_ids, b_assigned_students_data = (
                best_assignment.get_assigned_module_students()
            )
            if module_ids == []:
                module_ids = b_module_ids
                assigned_students_data = b_assigned_students_data
            else:
                for m_id in b_module_ids:
                    m_idx = b_module_ids.index(m_id)
                    assigned_students_data[m_idx] = pd.concat([assigned_students_data[m_idx], b_assigned_students_data[m_idx]], axis=0)

            b_assignment_summary_df = (
                best_assignment.get_all_assigned_modules(module_df)
            )
            assignment_summary_df = pd.concat([assignment_summary_df, b_assignment_summary_df], axis=0)

            b_excess_requests_df = (
                best_assignment.get_excess_module_requests()
            )
            excess_requests_df = pd.concat([excess_requests_df, b_excess_requests_df], axis=0)

        module_metadata_df = (
            ordered_best_assignments[-1].get_module_dataframe(module_df)
        )

        with ZipFile(
            zip_filename,
            "w"
        ) as zf:

            #
            # Individual module allocations
            #

            for m_idx, module_id in enumerate(module_ids):

                csv_buffer = io.BytesIO()

                assigned_students_data[m_idx].to_csv(
                    csv_buffer,
                    index=False
                )

                zf.writestr(
                    f"{module_id}.csv",
                    csv_buffer.getvalue()
                )

            #
            # Assignment summary
            #

            csv_buffer = io.BytesIO()

            assignment_summary_df.to_csv(
                csv_buffer,
                index=False
            )

            zf.writestr(
                "module_assignment_summary.csv",
                csv_buffer.getvalue()
            )

            #
            # Constraints summary
            #

            combined_df_semester_min = pd.DataFrame()
            combined_df_semester_max = pd.DataFrame()
            combined_df_group_min = pd.DataFrame()
            combined_df_group_max = pd.DataFrame()
            combined_df_total_credits = pd.DataFrame()
            combined_df_degree_programme_module_exclusions_satisfied = pd.DataFrame()
            combined_student_list_df = pd.DataFrame()

            for best_assignment in best_assignments_per_programme.values():


                semester_min_satisfied, semester_labels = (
                    best_assignment
                    .assignment_satisfies_minimum_credits_per_semester()
                )
                df_semester_min = pd.DataFrame(
                    semester_min_satisfied,
                    columns=[
                        f"min_credits_per_semester_satisfied_{s}"
                        for s in semester_labels
                    ]
                )
                combined_df_semester_min = pd.concat([combined_df_semester_min, df_semester_min], axis=0)
                

                semester_max_satisfied, semester_labels = (
                    best_assignment
                    .assignment_satisfies_maximum_credits_per_semester()
                )
                df_semester_max = pd.DataFrame(
                    semester_max_satisfied,
                    columns=[
                        f"max_credits_per_semester_not_exceeded_{s}"
                        for s in semester_labels
                    ]
                )
                combined_df_semester_max = pd.concat([combined_df_semester_max, df_semester_max], axis=0)
                

                group_min_satisfied, group_labels = (
                    best_assignment
                    .assignment_satisfies_minimum_credits_per_group()
                )
                df_group_min = pd.DataFrame(
                    group_min_satisfied,
                    columns=[
                        f"min_credits_per_group_satisfied_{g}"
                        for g in group_labels
                    ]
                )
                combined_df_group_min = pd.concat([combined_df_group_min, df_group_min], axis=0)


                group_max_satisfied, group_labels = (
                    best_assignment
                    .assignment_satisfies_maximum_credits_per_group()
                )
                df_group_max = pd.DataFrame(
                    group_max_satisfied,
                    columns=[
                        f"max_credits_per_group_not_exceeded_{g}"
                        for g in group_labels
                    ]
                )
                combined_df_group_max = pd.concat([combined_df_group_max, df_group_max], axis=0)
                

                total_credits_satisfied = best_assignment.get_assigned_credits_totals() == best_assignment._required_credits_per_student
                df_total_credits = pd.DataFrame(
                    total_credits_satisfied,
                    columns=[
                        "required_credits_total_satisfied"
                    ]
                )
                combined_df_total_credits = pd.concat([combined_df_total_credits, df_total_credits], axis=0)


                degree_programme_module_exclusions_satisfied = (
                    best_assignment.assignment_satisfies_degree_programme_module_exclusions()
                )
                df_degree_programme_module_exclusions_satisfied = pd.DataFrame(degree_programme_module_exclusions_satisfied, columns=["degree_programme_module_exclusions_satisfied"])
                combined_df_degree_programme_module_exclusions_satisfied = pd.concat([combined_df_degree_programme_module_exclusions_satisfied, df_degree_programme_module_exclusions_satisfied], axis=0)

                student_list_df = (
                    best_assignment.get_students_list()
                )
                combined_student_list_df = pd.concat([combined_student_list_df, student_list_df], axis=0)

            
            constraints_df = pd.concat(
                [
                    combined_student_list_df,
                    combined_df_semester_min,
                    combined_df_semester_max,
                    combined_df_group_min,
                    combined_df_group_max,
                    combined_df_total_credits,
                    combined_df_degree_programme_module_exclusions_satisfied
                ],
                axis=1,
            )

            csv_buffer = io.BytesIO()

            constraints_df.to_csv(
                csv_buffer,
                index=False
            )

            zf.writestr(
                "constraints_summary.csv",
                csv_buffer.getvalue()
            )

            #
            # Excess requests
            #

            csv_buffer = io.BytesIO()

            excess_requests_df.to_csv(
                csv_buffer,
                index=False
            )

            zf.writestr(
                "excess_module_requests.csv",
                csv_buffer.getvalue()
            )

            #
            # Module metadata
            #

            csv_buffer = io.BytesIO()

            module_metadata_df.to_csv(
                csv_buffer,
                index=False
            )

            zf.writestr(
                "module_metadata.csv",
                csv_buffer.getvalue()
            )

        print(
            "Download starting..."
        )

        # files.download(
        #     zip_filename
        # )

download_button.on_click(
    on_download_clicked
)

Button(button_style='warning', description='Download Results', icon='download', style=ButtonStyle())

Output()